# Deep learning recommender

Author: Masha Khanina

Reviewer: 

Description: making movie recommendations system by using two-tower model for retrieval + mlp with early crossing for ranking recommendations.

Infinity Labs R&D AI #5

In [268]:
import copy

import duckdb
import pandas as pd
import numpy as np
from numpy.typing import ArrayLike
from sklearn.preprocessing import OrdinalEncoder
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [269]:
with duckdb.connect("/home/maria-khanina/mkgit/movies_project/data/database.duckdb", read_only=True) as db_conn:
    ratings_df = db_conn.execute("""
        SELECT *
        FROM ratings
            INNER JOIN movies USING (movie_id)
            INNER JOIN users USING (user_id)
    """).df()

In [270]:
ratings_df.head()

,user_id,movie_id,rating,timestamp,title,genres,gender,age,occupation,zip_code
0,1,1193,5,978300760,One Flew Over the Cuckoo's Nest (1975),Drama,F,1,10,48067
1,1,661,3,978302109,James and the Giant Peach (1996),Animation|Children's|Musical,F,1,10,48067
2,1,914,3,978301968,My Fair Lady (1964),Musical|Romance,F,1,10,48067
3,1,3408,4,978300275,Erin Brockovich (2000),Drama,F,1,10,48067
4,1,2355,5,978824291,"Bug's Life, A (1998)",Animation|Children's|Comedy,F,1,10,48067


In [271]:
ratings_df["age"].value_counts()

age
25    395556
35    199003
18    183536
45     83633
50     72490
56     38780
1      27211
Name: count, dtype: int64

In [272]:
genres_encoded = ratings_df["genres"].str.get_dummies(sep="|")
genres_cols = genres_encoded.columns.to_list()
genres_num = len(genres_cols)
ratings_df = pd.concat([ratings_df, genres_encoded], axis=1)

In [273]:
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)

cat_cols = ["gender"]

ratings_df[cat_cols] = encoder.fit_transform(ratings_df[cat_cols]) + 1
ratings_df["occupation"] = ratings_df["occupation"] + 1

In [274]:
max_user = int(ratings_df["user_id"].max())
max_movie = int(ratings_df["movie_id"].max())
max_age = int(ratings_df["age"].max())
max_gender = int(ratings_df["gender"].max())
max_job = int(ratings_df["occupation"].max())

In [275]:
class Ratings(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        genres_cols: list[str],
        is_training: bool = True,
    ) -> None:
        self.user_ids = torch.tensor(df["user_id"].values, dtype=torch.long)
        self.movie_ids = torch.tensor(df["movie_id"].values, dtype=torch.long)
        self.ages = torch.tensor(df["age"].values, dtype=torch.long)
        self.genders = torch.tensor(df["gender"].values, dtype=torch.long)
        self.jobs = torch.tensor(df["occupation"].values, dtype=torch.long)
        self.ratings = torch.tensor(df["normalized_rating"].values, dtype=torch.float32)
        self.genres = torch.tensor(df[genres_cols].values, dtype=torch.float32)
        self.is_training = is_training

    def __len__(self) -> int:
        return len(self.user_ids)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        
        # user_id = int(row["user_id"])
        # if self.is_training and torch.rand(1).item() < 0.05:
            # user_id = 0
            
        return {
            "user_id": self.user_ids[idx],
            "movie_id": self.movie_ids[idx],
            "age": self.ages[idx],
            "gender": self.genders[idx],
            "job": self.jobs[idx],
            "genres": self.genres[idx], 
            "rating": self.ratings[idx],
        }

In [276]:
class BaseRanker(nn.Module):
    def __init__(
        self,
        users_num: int,
        movies_num: int,
        add_interactions: bool = False,
        lin_layers_dims: list[int] = [128, 64, 32],
        dropout_ratio=0.2,
        user_dim=16,
        movie_dim=16,
        extra_input_dim=0,
    ) -> None:
        super().__init__()
        
        self.add_interactions = add_interactions
        
        self.user_embedding = nn.Embedding(
            num_embeddings=users_num + 1,
            embedding_dim=user_dim,
        )
        self.movie_embedding = nn.Embedding(
            num_embeddings=movies_num + 1,
            embedding_dim=movie_dim,
        )

        input_dim = user_dim + movie_dim + extra_input_dim
        
        if self.add_interactions:
            assert user_dim == movie_dim
            input_dim += user_dim

        self.mlp = self._build_mlp(input_dim, lin_layers_dims, dropout_ratio)

    def _build_mlp(
        self,
        input_dim: int, 
        lin_layers_dims: list[int],
        dropout_ratio: float,
    ) -> nn.Sequential:
        layers = []
        curr_dim = input_dim
        
        for ind, dim in enumerate(lin_layers_dims):
            layers.append(nn.Linear(curr_dim, dim))
            layers.append(nn.SiLU())
            layers.append(nn.BatchNorm1d(dim))
            
            if ind == 0:
                layers.append(nn.Dropout(p=dropout_ratio))
                
            curr_dim = dim
            
        layers.append(nn.Linear(curr_dim, 1))
        return nn.Sequential(*layers)

    def _get_base_features(
        self,
        user_ids: torch.Tensor,
        movie_ids: torch.Tensor,
    ) -> list[torch.Tensor]:
        user_vector = self.user_embedding(user_ids)
        movie_vector = self.movie_embedding(movie_ids)
        
        features = [user_vector, movie_vector]
        
        if self.add_interactions:
            features.append(user_vector * movie_vector)
            
        return features

    def forward(
        self,
        user_ids: torch.Tensor,
        movie_ids: torch.Tensor,
    ) -> torch.Tensor:
        
        features = self._get_base_features(user_ids, movie_ids)
        cat_features = torch.cat(features, dim=1)
        
        return self.mlp(cat_features).squeeze()


class RankerWithFeatures(BaseRanker):
    def __init__(
        self,
        users_num: int,
        movies_num: int,
        ages_num: int,
        genders_num: int,
        jobs_num: int,
        genres_num: int,
        add_interactions: bool = False,
        lin_layers_dims: list[int] = [128, 64, 32],
        dropout_ratio=0.2,
        user_dim=16,
        movie_dim=16,
        age_dim=3,
        gender_dim=2,
        job_dim=4,
        genre_dim=4,
    ) -> None:
        
        extra_dim = age_dim + gender_dim + job_dim + genre_dim

        super().__init__(
            users_num=users_num,
            movies_num=movies_num,
            add_interactions=add_interactions,
            lin_layers_dims=lin_layers_dims,
            dropout_ratio=dropout_ratio,
            user_dim=user_dim,
            movie_dim=movie_dim,
            extra_input_dim=extra_dim,
        )

        self.job_embedding = nn.Embedding(
            num_embeddings=jobs_num + 1,
            embedding_dim=job_dim,
        )
        self.age_embedding = nn.Embedding(
            num_embeddings=ages_num + 1,
            embedding_dim=age_dim,
        )
        self.gender_embedding = nn.Embedding(
            num_embeddings=genders_num + 1,
            embedding_dim=gender_dim,
        )
        self.genre_projector = nn.Linear(genres_num, genre_dim, bias=False)

    def forward(
        self,
        user_ids: torch.Tensor,
        movie_ids: torch.Tensor,
        ages: torch.Tensor,
        genders: torch.Tensor,
        jobs: torch.Tensor,
        genres: torch.Tensor,
    ) -> torch.Tensor:
        
        features = self._get_base_features(user_ids, movie_ids)
        
        features.extend([
            self.age_embedding(ages),
            self.gender_embedding(genders),
            self.job_embedding(jobs),
            self.genre_projector(genres),
        ])
        
        cat_features = torch.cat(features, dim=1)
        
        return self.mlp(cat_features).squeeze()

In [ ]:
def _run_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    var_key_map: dict[str, str],
    criterion: nn.Module,
    device: torch.device,
    is_train: bool,
    optimizer: optim.Optimizer | None = None,
) -> float:
    if is_train:
        model.train()
    else:
        model.eval()
    
    epoch_loss = 0.0
    dataset_len = 0
    
    with torch.set_grad_enabled(is_train):
        for batch in dataloader:
            feature_tensors: dict[str, torch.tensor] = {}
            for var_name, col_name in var_key_map.items():
                feature_tensors[var_name] = batch[col_name].to(device)

            y_true = batch["rating"].to(device).float()
            y_pred = model(**feature_tensors)
            
            loss = criterion(y_pred, y_true)
            
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
            epoch_loss += loss.item() * feature_tensors["user_ids"].size(0)
            dataset_len += feature_tensors["user_ids"].size(0)
            
    return epoch_loss / dataset_len


def train_ranker(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    var_key_map: dict[str, str],
    epochs=10_000,
    learning_rate=0.001,
    patience=10,
    tol=1e-4,
    weight_decay=1e-5,
) -> None:
    """if torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")"""
    device = torch.device("cpu")
    model = model.to(device)
    
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    
    best_val_loss = np.inf
    to_beat_loss = np.inf
    no_improve_count = 0
    best_weights = None
    
    for epoch in range(epochs):
        avg_train_loss = _run_epoch(
            model=model,
            dataloader=train_loader,
            var_key_map=var_key_map,
            criterion=criterion,
            device=device,
            is_train=True,
            optimizer=optimizer,
        )
        
        avg_val_loss = _run_epoch(
            model=model,
            dataloader=val_loader,
            var_key_map=var_key_map,
            criterion=criterion,
            device=device,
            is_train=False,
        )
        
        print(
            f"Epoch {epoch+1}/{epochs}, "
            f"Train RMSE: {np.sqrt(avg_train_loss):.4f}, "
            f"Val RMSE: {np.sqrt(avg_val_loss):.4f}."
        )
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_weights = copy.deepcopy(model.state_dict())

        if avg_val_loss < (to_beat_loss - tol):
            to_beat_loss = avg_val_loss
            no_improve_count = 0
        else:
            no_improve_count += 1
            
        if no_improve_count >= patience:
            break
    
    if best_weights is not None:
        model.load_state_dict(best_weights)
        print(f"\nBest weights restored. Val RMSE: {np.sqrt(best_val_loss)}")

In [278]:
def per_var_split_mask(
        df_sorted: pd.DataFrame,
        var: str,
        max_test_ratio=0.1,
        test_max_per_var=8,
        test_min_per_var=3,
):
    """
        Prepare splitting for given dataframe such as each unique value of var 
        has max_test_ratio ratio of records in test set capped at test_max_per_var.
    """
    per_var_count = df_sorted[var].map(df_sorted[var].value_counts())
    per_var_ranks = df_sorted.groupby(var).cumcount(ascending=False)
    per_var_test_size = np.minimum(test_max_per_var, max_test_ratio * per_var_count)
    per_var_test_size = np.maximum(per_var_test_size, test_min_per_var)
    test_mask = per_var_ranks < per_var_test_size

    return test_mask

In [279]:
def per_var_temporal_split(
        X: pd.DataFrame,
        var="user_id",
        time_col="timestamp",
        max_test_ratio=0.1,
        test_max_per_var=8,
        test_min_per_var=3,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """
        Split X and y to train and test such as for each value of var
        all test records have greater time_col values than train records,
        and each value of var receives max_test_ratio of records in test,
        capped at test_max_per_var.
    """
    sort_indices = X.sort_values(by=[var, time_col]).index
    X_sorted = X.loc[sort_indices]
    test_mask = per_var_split_mask(
        X_sorted, 
        var, 
        max_test_ratio, 
        test_max_per_var,
        test_min_per_var,
    )
    return X_sorted[~test_mask], X_sorted[test_mask]

In [280]:
train_all_df, test_df = per_var_temporal_split(
    X=ratings_df,
    var="user_id",
    time_col="timestamp",
    max_test_ratio=0.1,
    test_max_per_var=8,
    test_min_per_var=3,
)

In [281]:
train_big_df, outer_val_df = per_var_temporal_split(
    train_all_df,
    var="user_id",
    time_col="timestamp",
    max_test_ratio=0.1,
    test_max_per_var=8,
    test_min_per_var=3,
)

In [282]:
train_df, val_df = per_var_temporal_split(
    train_big_df,
    var="user_id",
    time_col="timestamp",
    max_test_ratio=0.1,
    test_max_per_var=8,
    test_min_per_var=3,
)

In [283]:
rating_mean = train_df["rating"].mean()
users_rating_means = train_df.groupby("user_id")["rating"].mean()

for df in [train_df, val_df, outer_val_df, test_df]:
    df["user_mean"] = df["user_id"].map(users_rating_means).fillna(rating_mean)
    df["normalized_rating"] = df["rating"] - df["user_mean"]

In [286]:
var_key_map = {
    "user_ids": "user_id",
    "movie_ids": "movie_id",
    "ages": "age",
    "genders": "gender",
    "jobs": "job",
    "genres": "genres",
}

In [284]:
train_dataset = Ratings(train_df, genres_cols, is_training=True)
val_dataset = Ratings(val_df, genres_cols, is_training=False)

train_loader = DataLoader(
    train_dataset, 
    batch_size=1024, 
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=1024, 
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

ranker_with_interactions = RankerWithFeatures(
    users_num=max_user,
    movies_num=max_movie,
    ages_num=max_age,
    genders_num=max_gender,
    jobs_num=max_job,
    genres_num=genres_num,
    add_interactions=True,
)

train_ranker(
    model=ranker_with_interactions,
    train_loader=train_loader,
    val_loader=val_loader,
    var_key_map=var_key_map,
    epochs=10_000,
    learning_rate=0.002,
)

Epoch 1/10000, Train RMSE: 0.9719, Val RMSE: 0.9679.
Epoch 2/10000, Train RMSE: 0.9237, Val RMSE: 0.9554.
Epoch 3/10000, Train RMSE: 0.9103, Val RMSE: 0.9472.
Epoch 4/10000, Train RMSE: 0.8984, Val RMSE: 0.9391.
Epoch 5/10000, Train RMSE: 0.8895, Val RMSE: 0.9336.
Epoch 6/10000, Train RMSE: 0.8834, Val RMSE: 0.9290.
Epoch 7/10000, Train RMSE: 0.8784, Val RMSE: 0.9275.
Epoch 8/10000, Train RMSE: 0.8741, Val RMSE: 0.9257.
Epoch 9/10000, Train RMSE: 0.8706, Val RMSE: 0.9214.
Epoch 10/10000, Train RMSE: 0.8671, Val RMSE: 0.9196.
Epoch 11/10000, Train RMSE: 0.8639, Val RMSE: 0.9183.
Epoch 12/10000, Train RMSE: 0.8606, Val RMSE: 0.9167.
Epoch 13/10000, Train RMSE: 0.8573, Val RMSE: 0.9178.
Epoch 14/10000, Train RMSE: 0.8543, Val RMSE: 0.9159.
Epoch 15/10000, Train RMSE: 0.8515, Val RMSE: 0.9166.
Epoch 16/10000, Train RMSE: 0.8488, Val RMSE: 0.9155.
Epoch 17/10000, Train RMSE: 0.8461, Val RMSE: 0.9143.
Epoch 18/10000, Train RMSE: 0.8431, Val RMSE: 0.9138.
Epoch 19/10000, Train RMSE: 0.8408, V

In [ ]:
big_ranker_with_features = RankerWithFeatures(
    users_num=max_user,
    movies_num=max_movie,
    ages_num=max_age,
    genders_num=max_gender,
    jobs_num=max_job,
    genres_num=genres_num,
    add_interactions=False,
    dropout_ratio=0,
    user_dim=32,
    movie_dim=32,
    age_dim=4,
    gender_dim=1,
    job_dim=8,
    genre_dim=8,
)


train_ranker(
    model=big_ranker_with_features,
    train_loader=train_loader,
    val_loader=val_loader,
    var_key_map=var_key_map,
    epochs=10_000,
    learning_rate=0.002,
)

In [285]:
ranker_with_features = RankerWithFeatures(
    users_num=max_user,
    movies_num=max_movie,
    ages_num=max_age,
    genders_num=max_gender,
    jobs_num=max_job,
    genres_num=genres_num,
    add_interactions=False,
)

train_ranker(
    model=ranker_with_features,
    train_loader=train_loader,
    val_loader=val_loader,
    var_key_map=var_key_map,
    epochs=10_000,
    learning_rate=0.002,
)

NameError: name 'var_key_map' is not defined

In [ ]:
base_ranker = BaseRanker(
    users_num=max_user,
    movies_num=max_movie,
)

train_ranker(
    model=base_ranker,
    train_loader=train_loader,
    val_loader=val_loader,
    var_key_map={
        "user_ids": "user_id", 
        "movie_ids": "movie_id",
    },
    epochs=10_000,
    learning_rate=0.002,
)

Epoch 1/10000, Train RMSE: 0.9715, Val RMSE: 0.9703.
Epoch 2/10000, Train RMSE: 0.9282, Val RMSE: 0.9619.
Epoch 3/10000, Train RMSE: 0.9167, Val RMSE: 0.9507.
Epoch 4/10000, Train RMSE: 0.9033, Val RMSE: 0.9428.
Epoch 5/10000, Train RMSE: 0.8937, Val RMSE: 0.9355.
Epoch 6/10000, Train RMSE: 0.8878, Val RMSE: 0.9322.
Epoch 7/10000, Train RMSE: 0.8836, Val RMSE: 0.9263.
Epoch 8/10000, Train RMSE: 0.8805, Val RMSE: 0.9240.
Epoch 9/10000, Train RMSE: 0.8778, Val RMSE: 0.9221.
Epoch 10/10000, Train RMSE: 0.8755, Val RMSE: 0.9207.
Epoch 11/10000, Train RMSE: 0.8735, Val RMSE: 0.9199.
Epoch 12/10000, Train RMSE: 0.8716, Val RMSE: 0.9188.
Epoch 13/10000, Train RMSE: 0.8702, Val RMSE: 0.9184.
Epoch 14/10000, Train RMSE: 0.8686, Val RMSE: 0.9180.
Epoch 15/10000, Train RMSE: 0.8670, Val RMSE: 0.9178.
Epoch 16/10000, Train RMSE: 0.8653, Val RMSE: 0.9180.
Epoch 17/10000, Train RMSE: 0.8638, Val RMSE: 0.9171.
Epoch 18/10000, Train RMSE: 0.8626, Val RMSE: 0.9176.
Epoch 19/10000, Train RMSE: 0.8608, V